## Basic prompting

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from langchain_groq import ChatGroq
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
)

response = model.invoke(
    [HumanMessage(content="What's the capital of the Moon?")]
)
response

AIMessage(content='There isn’t one. The Moon is a natural satellite, not a country or political entity, so it has no government, no administrative divisions, and therefore no capital city. If you’re thinking of future lunar colonies or space‑faring nations, any “capital” would be a human‑made designation that hasn’t been established yet. In short, the Moon itself has no capital.', additional_kwargs={'reasoning_content': 'The user asks: "What\'s the capital of the Moon?" This is a trick question. The Moon has no capital. It\'s a fictional or rhetorical question. The answer: The Moon has no capital; it\'s not a country. Could mention that it\'s a celestial body, no government. Could mention that there is no capital. Could mention that if we talk about lunar colonies, no official capital. So answer: The Moon has no capital. Provide explanation.'}, response_metadata={'token_usage': {'completion_tokens': 178, 'prompt_tokens': 78, 'total_tokens': 256, 'completion_time': 0.18559942, 'completi

In [9]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(model=model)

question = HumanMessage(content="What's the capital of the moon?")

response = agent.invoke(
    {"messages": [question]}
)

print(response["messages"][-1].content)

The Moon doesn’t have a capital—there’s no government or city that serves as a capital on the lunar surface. It’s just a natural satellite of Earth, not a country or state.


In [11]:
system_prompt = "You are a science fiction writer, create a capital city at the users request."

scifi_agent = create_agent(
    model=model,  # reuse the existing ChatGroq instance
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response["messages"][-1].content)

**Capital of the Moon: Selene Prime**

*Location:*  
Selene Prime sits on the far side of the Moon, nestled within the vast, basaltic plain of the Mare Tranquillitatis. The city’s core is anchored in a natural basin that has been sealed with a transparent, self‑healing polymer shell, creating a breathable, pressurized habitat that shields its inhabitants from micrometeorites and solar radiation.

*Architecture:*  
The city’s skyline is a blend of sleek, low‑profile domes and spiraling towers that mimic the lunar regolith’s layered strata. Each dome is a self‑contained ecosystem, with hydroponic gardens, algae bioreactors, and a central “sky‑garden” that offers a view of the Earthrise. The towers are constructed from a composite of lunar regolith and recycled titanium, giving them a silvery, almost translucent appearance under the weak lunar light.

*Governance:*  
Selene Prime is governed by the Lunar Assembly, a council elected by the city’s residents. The Assembly operates on a conse

## Few-shot examples

In [12]:
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

User: What is the capital of mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia

"""

scifi_agent = create_agent(
    model=model,  # reuse the existing ChatGroq instance
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

Scifi Writer: Lunaris


## Structured prompts

In [14]:
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

Please keep to the below structure.

Name: The name of the capital city

Location: Where it is based

Vibe: 2-3 words to describe its vibe

Economy: Main industries

"""

scifi_agent = create_agent(
    model=model,  # reuse the existing ChatGroq instance
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

Name: Selene City  
Location: Mare Tranquillitatis, near the lunar equator on the near side of the Moon  
Vibe: Sterile, futuristic, resilient  
Economy: Lunar mining, scientific research, space tourism, high‑tech manufacturing  


## Structured output

In [15]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel

class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

agent = create_agent(
    model=model,  # reuse the existing ChatGroq instance
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
    response_format=CapitalInfo
)

question = HumanMessage(content="What is the capital of The Moon?")

response = agent.invoke(
    {"messages": [question]}
)

response["structured_response"]

CapitalInfo(name='Selene City', location='Mare Tranquillitatis, near the lunar equator', vibe='Sterile, efficient, with a hint of isolation', economy='High-tech research and mining of helium-3 and rare minerals')

In [16]:
response["structured_response"].name

'Selene City'

In [17]:
capital_info = response["structured_response"]

capital_name = capital_info.name
capital_location = capital_info.location

print(f"{capital_name} is a city located at {capital_location}")

Selene City is a city located at Mare Tranquillitatis, near the lunar equator
